In [7]:
from git import Repo 
from langchain.text_splitter import Language, RecursiveCharacterTextSplitter
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI 
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser


import os 


## Clone the Git hub repo here


In [2]:
!mkdir -p test_repo

In [3]:
repo_path= "test_repo"
repo = Repo.clone_from("https://github.com/entbappy/End-to-End-Medical-Chatbot", to_path=repo_path)

## Load the data from the cloned folders

In [4]:
%pwd

'd:\\Real Time Source code analyzer\\research'

In [10]:
loader = GenericLoader.from_filesystem(repo_path,
                                      glob="**/*.*",
                                      suffixes=[".py"],
                                      parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
 )

In [11]:
documents = loader.load()

In [12]:
documents 

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask, render_template, jsonify, request\nfrom src.helper import download_hugging_face_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import ChatOpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import *\nimport os\n\napp = Flask(__name__)\n\n\nload_dotenv()\n\n\nPINECONE_API_KEY=os.environ.get(\'PINECONE_API_KEY\')\nOPENAI_API_KEY=os.environ.get(\'OPENAI_API_KEY\')\n\nos.environ["PINECONE_API_KEY"] = PINECONE_API_KEY\nos.environ["OPENAI_API_KEY"] = OPENAI_API_KEY\n\nembeddings = download_hugging_face_embeddings()\n\n\nindex_name = "medical-chatbot" \n# Embed each chunk and upsert the embeddings into your Pinecone index.\ndocsearch = Pin

In [13]:
len(documents)

6

## split the documents into chunks


In [14]:
document_spliter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON,
                                                                chunk_size=500, 
                                                                chunk_overlap=20)

In [15]:
text = document_spliter.split_documents(documents)

In [16]:
text

[Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='from flask import Flask, render_template, jsonify, request\nfrom src.helper import download_hugging_face_embeddings\nfrom langchain_pinecone import PineconeVectorStore\nfrom langchain_openai import ChatOpenAI\nfrom langchain.chains import create_retrieval_chain\nfrom langchain.chains.combine_documents import create_stuff_documents_chain\nfrom langchain_core.prompts import ChatPromptTemplate\nfrom dotenv import load_dotenv\nfrom src.prompt import *\nimport os\n\napp = Flask(__name__)\n\n\nload_dotenv()'),
 Document(metadata={'source': 'test_repo\\app.py', 'language': <Language.PYTHON: 'python'>}, page_content='load_dotenv()\n\n\nPINECONE_API_KEY=os.environ.get(\'PINECONE_API_KEY\')\nOPENAI_API_KEY=os.environ.get(\'OPENAI_API_KEY\')\n\nos.environ["PINECONE_API_KEY"] = PINECONE_API_KEY\nos.environ["OPENAI_API_KEY"] = OPENAI_API_KEY\n\nembeddings = download_hugging_face_embeddings()\n

In [17]:
len(text)

14

## Download the OpenAi Embedding

In [19]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [20]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## setting up vector databases

In [22]:
embeddings = OpenAIEmbeddings()

C:\Users\saiki\AppData\Local\Temp\ipykernel_10772\2497576997.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [24]:
vectordb = Chroma.from_documents(text,embedding=embeddings, persist_directory="./data")

ImportError: Could not import chromadb python package. Please install it with `pip install chromadb`.